# Filtering

In [1]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt

import warnings 
warnings.filterwarnings('ignore')

## Set input path

In [3]:
# Set path
base_path = '/stanley/WangLab/Data/Analyzed/2024-09-05-Mingrui-PFC-rep1/expr/2D_overlay'
# base_path = '/stanley/WangLab/Data/Analyzed/2024-11-18-Mingrui-PFC-rep2/expr'

# base_path = '/stanley/WangLab/Data/Analyzed/2024-10-05-Mingrui-HP-rep1/expr'
# base_path = '/stanley/WangLab/Data/Analyzed/2024-10-21-Mingrui-HP-rep2/expr'

# base_path = '/stanley/WangLab/Data/Analyzed/2024-09-23-Mingrui-ST-rep1/expr'
# base_path = '/stanley/WangLab/Data/Analyzed/2024-11-03-Mingrui-ST-rep2/expr'

input_path = base_path

out_path = os.path.join(input_path, 'filtered h5ad')
if not os.path.exists(out_path):
    os.mkdir(out_path)

### WT_STAR

In [4]:
sdata_wt = sc.read_h5ad(os.path.join(input_path, 'sample2_raw.h5ad'))

In [5]:
sc.pl.highest_expr_genes(sdata_wt, n_top=20)

In [6]:
# calculate pp metric
sc.pp.calculate_qc_metrics(sdata_wt, inplace=True)

# Calculate max count for each gene
sdata_wt.var['max_counts_sample'] = sdata_wt.X.max(axis=0)

In [7]:
# Total counts describe statistics
sdata_wt.obs['total_counts'].describe()

In [8]:
# max counts describe statistics
sdata_wt.var['max_counts_sample'].describe()

In [9]:
# mad threshold
from scipy import stats
n = 3
mad = stats.median_abs_deviation(sdata_wt.obs['log1p_total_counts'], scale=1)

lower_bd = sdata_wt.obs['log1p_total_counts'].median() - n*mad
upper_bd = sdata_wt.obs['log1p_total_counts'].median() + n*mad

print(lower_bd)
print(upper_bd)
print(np.expm1(lower_bd))
print(np.expm1(upper_bd))

In [10]:
# mad threshold
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(12,5))
sns.histplot(sdata_wt.obs['total_counts'], ax=axs[0])
axs[0].axvline(np.expm1(lower_bd), c='r')
axs[0].axvline(np.expm1(upper_bd), c='r')

sns.histplot(sdata_wt.obs['log1p_total_counts'], ax=axs[1])
axs[1].axvline(lower_bd, c='r')
axs[1].axvline(upper_bd, c='r')

# plt.savefig(os.path.join(fig_path, 'reads_filtering_threshold.pdf'))
plt.show()

In [11]:
# Total counts describe statistics
ncell_left = sdata_wt.obs.loc[(sdata_wt.obs['total_counts'] >= np.expm1(lower_bd)) & (sdata_wt.obs['total_counts'] <= np.expm1(upper_bd)), 'total_counts'].shape
median_counts = sdata_wt.obs.loc[(sdata_wt.obs['total_counts'] >= np.expm1(lower_bd)) & (sdata_wt.obs['total_counts'] <= np.expm1(upper_bd)), 'total_counts'].median()

print(f'With current threshold, there are {ncell_left[0]} cells left and median counts per cell is {median_counts}')

In [12]:
# Filter gene by max counts 
sdata_wt.var['detected_sample'] = sdata_wt.var['max_counts_sample'] > 2
sdata_wt.var['highly_variable_sample'] = sdata_wt.var['max_counts_sample'] > 2
print(sdata_wt.var['detected_sample'].sum())

In [13]:
volume_upper_bd = np.percentile(sdata_wt.obs['volume'],99.5)
print(volume_upper_bd)

In [14]:
fig, axs = plt.subplots(figsize=(7,5))
sns.histplot(sdata_wt.obs['volume'])
axs.axvline(volume_upper_bd, c='r')

In [15]:
# Filtration (cell)
sc.pp.filter_cells(sdata_wt, min_genes=10)
sc.pp.filter_cells(sdata_wt, min_counts=np.expm1(lower_bd))
sc.pp.filter_cells(sdata_wt, max_counts=np.expm1(upper_bd))
sdata_wt = sdata_wt[sdata_wt.obs['volume'] <= volume_upper_bd]

sdata_wt.layers['raw'] = sdata_wt.X.copy()
sdata_wt.X.shape

In [16]:
from datetime import datetime
date = datetime.today().strftime('%Y-%m-%d')
sdata_wt.write_h5ad(f"{out_path}/{date}-WT_STAR-3mad-filtered.h5ad")

### HET_STAR

In [17]:
sdata_het=sc.read_h5ad(os.path.join(input_path, 'sample4_raw.h5ad'))

In [18]:
sc.pl.highest_expr_genes(sdata_het, n_top=20)

In [19]:
# calculate pp metric
sc.pp.calculate_qc_metrics(sdata_het, inplace=True)

# Calculate max count for each gene
sdata_het.var['max_counts_sample'] = sdata_het.X.max(axis=0)

In [20]:
# Total counts describe statistics
sdata_het.obs['total_counts'].describe()

In [21]:
# max counts describe statistics
sdata_het.var['max_counts_sample'].describe()

In [22]:
# mad threshold
from scipy import stats
n = 3
mad = stats.median_abs_deviation(sdata_het.obs['log1p_total_counts'], scale=1)

lower_bd = sdata_het.obs['log1p_total_counts'].median() - n*mad
upper_bd = sdata_het.obs['log1p_total_counts'].median() + n*mad

print(lower_bd)
print(upper_bd)
print(np.expm1(lower_bd))
print(np.expm1(upper_bd))

In [23]:
# mad threshold
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(12,5))
sns.histplot(sdata_het.obs['total_counts'], ax=axs[0])
axs[0].axvline(np.expm1(lower_bd), c='r')
axs[0].axvline(np.expm1(upper_bd), c='r')

sns.histplot(sdata_het.obs['log1p_total_counts'], ax=axs[1])
axs[1].axvline(lower_bd, c='r')
axs[1].axvline(upper_bd, c='r')

# plt.savefig(os.path.join(fig_path, 'reads_filtering_threshold.pdf'))
plt.show()

In [24]:
# Total counts describe statistics
ncell_left = sdata_het.obs.loc[(sdata_het.obs['total_counts'] >= np.expm1(lower_bd)) & (sdata_het.obs['total_counts'] <= np.expm1(upper_bd)), 'total_counts'].shape
median_counts = sdata_het.obs.loc[(sdata_het.obs['total_counts'] >= np.expm1(lower_bd)) & (sdata_het.obs['total_counts'] <= np.expm1(upper_bd)), 'total_counts'].median()

print(f'With current threshold, there are {ncell_left[0]} cells left and median counts per cell is {median_counts}')

In [25]:
# Filter gene by max counts 
sdata_het.var['detected_sample'] = sdata_het.var['max_counts_sample'] > 2
sdata_het.var['highly_variable_sample'] = sdata_het.var['max_counts_sample'] > 2
print(sdata_het.var['detected_sample'].sum())

In [26]:
volume_upper_bd = np.percentile(sdata_het.obs['volume'],99.5)
print(volume_upper_bd)

In [27]:
fig, axs = plt.subplots(figsize=(7,5))
sns.histplot(sdata_het.obs['volume'])
axs.axvline(volume_upper_bd, c='r');

In [28]:
# Filtration (cell)
sc.pp.filter_cells(sdata_het, min_genes=10)
sc.pp.filter_cells(sdata_het, min_counts=np.expm1(lower_bd))
sc.pp.filter_cells(sdata_het, max_counts=np.expm1(upper_bd))
sdata_het = sdata_het[sdata_het.obs['volume'] <= volume_upper_bd]

sdata_het.layers['raw'] = sdata_het.X.copy()
sdata_het.X.shape

In [29]:
from datetime import datetime
date = datetime.today().strftime('%Y-%m-%d')
sdata_het.write_h5ad(f"{out_path}/{date}-HET_STAR-3mad-filtered.h5ad")

## RIBO


### WT_RIBO

In [30]:
rdata_wt=sc.read_h5ad(os.path.join(input_path, 'sample1_raw.h5ad'))

In [31]:
sc.pl.highest_expr_genes(rdata_wt, n_top=20)

In [32]:
# calculate pp metric
sc.pp.calculate_qc_metrics(rdata_wt, inplace=True)

# Calculate max count for each gene
rdata_wt.var['max_counts_sample'] = rdata_wt.X.max(axis=0)

In [33]:
# Total counts describe statistics
rdata_wt.obs['total_counts'].describe()

In [34]:
# max counts describe statistics
rdata_wt.var['max_counts_sample'].describe()

In [35]:
# mad threshold
from scipy import stats
n = 3
mad = stats.median_abs_deviation(rdata_wt.obs['log1p_total_counts'], scale=1)

lower_bd = rdata_wt.obs['log1p_total_counts'].median() - n*mad
upper_bd = rdata_wt.obs['log1p_total_counts'].median() + n*mad

print(lower_bd)
print(upper_bd)
print(np.expm1(lower_bd))
print(np.expm1(upper_bd))

In [36]:
# mad threshold
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(12,5))
sns.histplot(rdata_wt.obs['total_counts'], ax=axs[0])
axs[0].axvline(np.expm1(lower_bd), c='r')
axs[0].axvline(np.expm1(upper_bd), c='r')

sns.histplot(rdata_wt.obs['log1p_total_counts'], ax=axs[1])
axs[1].axvline(lower_bd, c='r')
axs[1].axvline(upper_bd, c='r')

# plt.savefig(os.path.join(fig_path, 'reads_filtering_threshold.pdf'))
plt.show()

In [37]:
# Total counts describe statistics
ncell_left = rdata_wt.obs.loc[(rdata_wt.obs['total_counts'] >= np.expm1(lower_bd)) & (rdata_wt.obs['total_counts'] <= np.expm1(upper_bd)), 'total_counts'].shape
median_counts = rdata_wt.obs.loc[(rdata_wt.obs['total_counts'] >= np.expm1(lower_bd)) & (rdata_wt.obs['total_counts'] <= np.expm1(upper_bd)), 'total_counts'].median()

print(f'With current threshold, there are {ncell_left[0]} cells left and median counts per cell is {median_counts}')

In [38]:
# Filter gene by max counts 
rdata_wt.var['detected_sample'] = rdata_wt.var['max_counts_sample'] > 2
rdata_wt.var['highly_variable_sample'] = rdata_wt.var['max_counts_sample'] > 2
print(rdata_wt.var['detected_sample'].sum())

In [39]:
volume_upper_bd = np.percentile(rdata_wt.obs['volume'],99.5)
print(volume_upper_bd)

In [40]:
fig, axs = plt.subplots(figsize=(7,5))
sns.histplot(rdata_wt.obs['volume'])
axs.axvline(volume_upper_bd, c='r');

In [41]:
# Filtration (cell)
sc.pp.filter_cells(rdata_wt, min_genes=10)
sc.pp.filter_cells(rdata_wt, min_counts=np.expm1(lower_bd))
sc.pp.filter_cells(rdata_wt, max_counts=np.expm1(upper_bd))
rdata_wt = rdata_wt[rdata_wt.obs['volume'] <= volume_upper_bd]

rdata_wt.layers['raw'] = rdata_wt.X.copy()
rdata_wt.X.shape

In [42]:
from datetime import datetime
date = datetime.today().strftime('%Y-%m-%d')
rdata_wt.write_h5ad(f"{out_path}/{date}-WT_RIBO-3mad-filtered.h5ad")

### HET_RIBO

In [43]:
rdata_het=sc.read_h5ad(os.path.join(input_path, 'sample3_raw.h5ad'))

In [44]:
sc.pl.highest_expr_genes(rdata_het, n_top=20)

In [45]:
# calculate pp metric
sc.pp.calculate_qc_metrics(rdata_het, inplace=True)

# Calculate max count for each gene
rdata_het.var['max_counts_sample'] = rdata_het.X.max(axis=0)

In [46]:
# Total counts describe statistics
rdata_het.obs['total_counts'].describe()

In [47]:
# max counts describe statistics
rdata_het.var['max_counts_sample'].describe()

In [48]:
# mad threshold
from scipy import stats
n = 3
mad = stats.median_abs_deviation(rdata_het.obs['log1p_total_counts'], scale=1)

lower_bd = rdata_het.obs['log1p_total_counts'].median() - n*mad
upper_bd = rdata_het.obs['log1p_total_counts'].median() + n*mad

print(lower_bd)
print(upper_bd)
print(np.expm1(lower_bd))
print(np.expm1(upper_bd))

In [49]:
# mad threshold
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(12,5))
sns.histplot(rdata_het.obs['total_counts'], ax=axs[0])
axs[0].axvline(np.expm1(lower_bd), c='r')
axs[0].axvline(np.expm1(upper_bd), c='r')

sns.histplot(rdata_het.obs['log1p_total_counts'], ax=axs[1])
axs[1].axvline(lower_bd, c='r')
axs[1].axvline(upper_bd, c='r')

# plt.savefig(os.path.join(fig_path, 'reads_filtering_threshold.pdf'))
plt.show()

In [50]:
# Total counts describe statistics
ncell_left = rdata_het.obs.loc[(rdata_het.obs['total_counts'] >= np.expm1(lower_bd)) & (rdata_het.obs['total_counts'] <= np.expm1(upper_bd)), 'total_counts'].shape
median_counts = rdata_het.obs.loc[(rdata_het.obs['total_counts'] >= np.expm1(lower_bd)) & (rdata_het.obs['total_counts'] <= np.expm1(upper_bd)), 'total_counts'].median()

print(f'With current threshold, there are {ncell_left[0]} cells left and median counts per cell is {median_counts}')


In [51]:
# Filter gene by max counts 
rdata_het.var['detected_sample'] = rdata_het.var['max_counts_sample'] > 2
rdata_het.var['highly_variable_sample'] = rdata_het.var['max_counts_sample'] > 2
print(rdata_het.var['detected_sample'].sum())

In [52]:
volume_upper_bd = np.percentile(rdata_het.obs['volume'],99.5)
print(volume_upper_bd)

In [53]:
fig, axs = plt.subplots(figsize=(7,5))
sns.histplot(rdata_het.obs['volume'])
axs.axvline(volume_upper_bd, c='r');

In [54]:
# Filtration (cell)
sc.pp.filter_cells(rdata_het, min_genes=10)
sc.pp.filter_cells(rdata_het, min_counts=np.expm1(lower_bd))
sc.pp.filter_cells(rdata_het, max_counts=np.expm1(upper_bd))
rdata_het = rdata_het[rdata_het.obs['volume'] <= volume_upper_bd]

rdata_het.layers['raw'] = rdata_het.X.copy()
rdata_het.X.shape

In [55]:
from datetime import datetime
date = datetime.today().strftime('%Y-%m-%d')
rdata_het.write_h5ad(f"{out_path}/{date}-HET_RIBO-3mad-filtered.h5ad")

## Correlation

In [47]:
# Correlation between two protocol
rdata_vector = np.log2(np.array(rdata_het.X.sum(axis=0)))
sdata_vector = np.log2(np.array(sdata_het.X.sum(axis=0)))

from scipy import stats
p_corr = stats.pearsonr(rdata_vector, sdata_vector)

corre_df = pd.DataFrame({'RIBOmap': rdata_vector, 'STARmap': sdata_vector})
g = sns.lmplot(x='RIBOmap', y='STARmap', data=corre_df, scatter_kws={'s': 1}, line_kws={'color': 'r'})
g.set_axis_labels('RIBOmap - log2(total counts)', 'STARmap - log2(total counts)')
plt.title(f"Pearson's correlation coefficient: {round(p_corr[0], 3)}")
#plt.savefig(os.path.join(fig_path, 'correlation_ribomap_starmap_hom_3mad.pdf'))
plt.show()